# Frozen Feature kNN Model Comparison

Run frozen DINO-style backbones on ChestMNIST and evaluate kNN on the resulting feature space. This official comparison notebook does not reuse cached feature banks: every timed run extracts features, runs kNN, and reports the work it actually performed. Compact CSV/JSON result tables can be saved; heavy feature tensors are not saved here.


In [ ]:
import os

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

print("Project root:", project_root)


In [ ]:
import gc
import time
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from packaging.version import Version
from torch.utils.data import DataLoader

MIN_TRANSFORMERS_FOR_DINOV3 = Version("4.57.2")
if Version(transformers.__version__) < MIN_TRANSFORMERS_FOR_DINOV3:
    raise RuntimeError(
        "DINOv3 requires transformers>=4.57.2,<5 in this project. "
        "Run: python -m pip install -r ../requirements.txt "
        f"(current transformers={transformers.__version__})."
    )

from src.data import get_small_data
from src.model import extract_feature_bank, get_dino_backbone, model_metadata
from src.results import make_run_dir, save_knn_outputs, validate_save_request
from src.train import (
    DEFAULT_KNN_FEWSHOT_SETTINGS,
    filter_knn_fewshot_settings,
    make_knn_search_settings,
    run_knn_fewshot_experiment,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("Transformers:", transformers.__version__)

def tensor_storage_mb(*tensors):
    return float(sum(tensor.numel() * tensor.element_size() for tensor in tensors) / 1024**2)

def current_peak_gpu_memory_mb(device):
    if torch.device(device).type == "cuda" and torch.cuda.is_available():
        return float(torch.cuda.max_memory_allocated(device) / 1024**2)
    return None


## Config


In [ ]:
dataset_name = "chestmnist"
data_fraction = 1.00
dataset_seed = 0
save_outputs = True

# k changes the neighbor vote, so it is part of the kNN evaluation grid.
# Thresholds are swept after each probability set without rerunning neighbor search.
knn_k_values = [1, 3, 5, 10, 20, 50, 100, 200, 500, 1000, 2000]
thresholds = [0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5]
selection_metric = "f1_macro_mean"
knn_batch_size = 256
seeds = list(range(10))
output_root = project_root / "outputs"

model_configs = [
    {
        "run_name": "dinov2_small_224",
        "model_name": "facebook/dinov2-small",
        "image_size": 224,
        "batch_size": 32,
    },
    {
        "run_name": "dinov2_large_224",
        "model_name": "facebook/dinov2-large",
        "image_size": 224,
        "batch_size": 16,
    },
    {
        "run_name": "dinov3_large_224",
        "model_name": "facebook/dinov3-vitl16-pretrain-lvd1689m",
        "image_size": 224,
        "batch_size": 16,
    },
    {
        "run_name": "stanford_dinov2_xray_224",
        "model_name": "StanfordAIMI/dinov2-base-xray-224",
        "image_size": 224,
        "batch_size": 16,
    },
    {
        "run_name": "rad_dino_518",
        "model_name": "microsoft/rad-dino",
        "image_size": 518,
        "batch_size": 8,
    },
]

validate_save_request(save_outputs, data_fraction)
knn_search_settings = make_knn_search_settings(DEFAULT_KNN_FEWSHOT_SETTINGS, knn_k_values)
models_to_run = model_configs
print("candidate kNN settings before data-size filtering:", len(knn_search_settings))
print("thresholds per kNN probability set:", len(thresholds))
models_to_run


## Run Experiments


In [ ]:
all_summaries = []
all_per_class_full = []
all_metadata = []

for model_config in tqdm(models_to_run, desc="Models", leave=True):
    run_start = time.perf_counter()
    run_name = model_config["run_name"]
    model_name = model_config["model_name"]
    image_size = model_config["image_size"]
    batch_size = model_config["batch_size"]
    run_dir = make_run_dir(
        output_root,
        dataset_name,
        run_name,
        data_fraction=data_fraction,
        dataset_seed=dataset_seed,
    )

    print("\n" + "=" * 80)
    print(run_name)
    print(model_name)

    data_start = time.perf_counter()
    train_loader, val_loader, class_names = get_small_data(
        batch_size=batch_size,
        image_size=image_size,
        data_fraction=data_fraction,
        seed=dataset_seed,
    )
    data_load_seconds = time.perf_counter() - data_start

    train_feature_loader = DataLoader(
        train_loader.dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    val_feature_loader = DataLoader(
        val_loader.dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    context = {
        "dataset_name": dataset_name,
        "adaptation_method": "frozen_feature_knn",
        "run_name": run_name,
        "model_name": model_name,
        "image_size": image_size,
    }

    model_start = time.perf_counter()
    model = get_dino_backbone(model_name).to(device)
    backbone_metadata = model_metadata(model)
    model_load_seconds = time.perf_counter() - model_start
    print("total params:", f"{backbone_metadata['total_params']:,}")
    print("trainable params:", f"{backbone_metadata['trainable_params']:,}")

    feature_bank, feature_metadata = extract_feature_bank(
        model,
        train_feature_loader,
        val_feature_loader,
        device,
        class_names=class_names,
    )
    feature_storage_mb = tensor_storage_mb(
        feature_bank["train_features"],
        feature_bank["val_features"],
    )

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    settings = filter_knn_fewshot_settings(
        knn_search_settings,
        feature_bank["train_features"].shape[0],
    )
    print("kNN settings after data-size filtering:", len(settings))
    print("metric rows before seed averaging:", len(settings) * len(thresholds))
    knn_outputs = run_knn_fewshot_experiment(
        train_features=feature_bank["train_features"],
        train_labels=feature_bank["train_labels"],
        val_features=feature_bank["val_features"],
        val_labels=feature_bank["val_labels"],
        class_names=feature_bank["class_names"],
        settings=settings,
        seeds=seeds,
        thresholds=thresholds,
        batch_size=knn_batch_size,
        device=device,
        context=context,
        show_progress=True,
    )

    feature_extraction_seconds = float(feature_metadata["feature_extraction_seconds"])
    adaptation_seconds = 0.0
    inference_or_eval_seconds = float(knn_outputs["metadata"]["knn_query_seconds"])
    threshold_search_seconds = float(knn_outputs["metadata"]["threshold_search_seconds"])
    efficiency_counted_seconds = float(
        feature_extraction_seconds
        + adaptation_seconds
        + inference_or_eval_seconds
        + threshold_search_seconds
    )
    actual_wall_seconds = float(time.perf_counter() - run_start)

    metadata = {
        **context,
        "data_fraction": data_fraction,
        "dataset_seed": dataset_seed,
        "batch_size": batch_size,
        "knn_batch_size": knn_batch_size,
        "thresholds": thresholds,
        "selection_metric": selection_metric,
        "num_knn_settings": int(knn_outputs["metadata"]["num_knn_settings"]),
        "num_reference_set_groups": int(knn_outputs["metadata"]["num_reference_set_groups"]),
        "num_thresholds": int(knn_outputs["metadata"]["num_thresholds"]),
        "num_total_metric_rows": int(len(knn_outputs["runs"])),
        "n_train": int(feature_bank["train_features"].shape[0]),
        "n_val": int(feature_bank["val_features"].shape[0]),
        "device": device,
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        **backbone_metadata,
        "trainable_param_fraction": (
            float(backbone_metadata["trainable_params"] / backbone_metadata["total_params"])
            if backbone_metadata["total_params"]
            else float("nan")
        ),
        "feature_dim": int(feature_bank["train_features"].shape[1]),
        "feature_storage_mb": feature_storage_mb,
        "train_feature_seconds": float(feature_metadata["train_feature_seconds"]),
        "val_feature_seconds": float(feature_metadata["val_feature_seconds"]),
        "feature_extraction_seconds": feature_extraction_seconds,
        "adaptation_seconds": adaptation_seconds,
        "inference_or_eval_seconds": inference_or_eval_seconds,
        "threshold_search_seconds": threshold_search_seconds,
        "efficiency_counted_seconds": efficiency_counted_seconds,
        "actual_wall_seconds": actual_wall_seconds,
        "peak_gpu_memory_mb": current_peak_gpu_memory_mb(device) or feature_metadata.get("peak_gpu_memory_mb"),
        "shortcut_used": False,
    }

    if save_outputs:
        save_knn_outputs(
            run_dir,
            runs_df=knn_outputs["runs"],
            summary_df=knn_outputs["summary"],
            per_class_full_df=knn_outputs["per_class_full"],
            metadata=metadata,
        )

    all_summaries.append(knn_outputs["summary"])
    all_per_class_full.append(knn_outputs["per_class_full"])
    all_metadata.append(metadata)

    del feature_bank
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

combined_summary_df = pd.concat(all_summaries, ignore_index=True)
combined_per_class_full_df = pd.concat(all_per_class_full, ignore_index=True)
metadata_df = pd.DataFrame(all_metadata)
combined_summary_df


## Reporting Policy

Use one validation-selected k/threshold row per model and reference-set size for headline plots. The full grid remains in `combined_summary_df` and saved CSVs for audit, but selected rows avoid treating searched hyperparameters as repeated estimates.


In [ ]:
summary_sort_cols = [selection_metric, "mean_auc_mean", "f1_micro_mean", "mean_accuracy_mean"]
selected_summary_df = (
    combined_summary_df.sort_values(
        ["run_name", "setting", *summary_sort_cols],
        ascending=[True, True, False, False, False, False],
    )
    .groupby(["run_name", "setting"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

selected_full_keys_df = selected_summary_df[selected_summary_df["setting"] == "full"][
    ["run_name", "setting", "n_train", "k", "threshold"]
].drop_duplicates()
selected_per_class_df = combined_per_class_full_df.merge(
    selected_full_keys_df,
    on=["run_name", "setting", "n_train", "k", "threshold"],
    how="inner",
)

selected_summary_df


## Compare Full Reference Rows


In [ ]:
full_rows_df = selected_summary_df[selected_summary_df["setting"] == "full"].copy()
full_rows_df[
    [
        "run_name",
        "model_name",
        "image_size",
        "n_train",
        "k",
        "threshold",
        "mean_auc_mean",
        "f1_macro_mean",
        "f1_micro_mean",
        "mean_accuracy_mean",
        "exact_match_accuracy_mean",
    ]
]


In [ ]:
metadata_cols = [
    "run_name",
    "model_name",
    "image_size",
    "total_params",
    "trainable_params",
    "trainable_param_fraction",
    "feature_dim",
    "selection_metric",
    "num_knn_settings",
    "num_reference_set_groups",
    "num_thresholds",
    "num_total_metric_rows",
    "feature_storage_mb",
    "feature_extraction_seconds",
    "adaptation_seconds",
    "inference_or_eval_seconds",
    "threshold_search_seconds",
    "efficiency_counted_seconds",
    "actual_wall_seconds",
    "peak_gpu_memory_mb",
    "shortcut_used",
]
metadata_df[metadata_cols]


## Visualize Model Comparison


In [ ]:
display_names = {
    "dinov2_small_224": "DINOv2-S 224",
    "dinov2_large_224": "DINOv2-L 224",
    "dinov3_large_224": "DINOv3-L 224",
    "stanford_dinov2_xray_224": "Stanford X-ray DINOv2-B 224",
    "rad_dino_518": "RAD-DINO 518",
}

model_groups = {
    "dinov2_small_224": "Generic DINOv2",
    "dinov2_large_224": "Generic DINOv2",
    "dinov3_large_224": "Generic DINOv3",
    "stanford_dinov2_xray_224": "Medical/X-ray",
    "rad_dino_518": "Medical/X-ray",
}

group_colors = {
    "Generic DINOv2": "tab:blue",
    "Generic DINOv3": "tab:purple",
    "Medical/X-ray": "tab:green",
}

plot_summary_df = selected_summary_df.copy()
plot_summary_df["display_name"] = plot_summary_df["run_name"].map(display_names).fillna(plot_summary_df["run_name"].str.replace("_", " "))
plot_summary_df["model_group"] = plot_summary_df["run_name"].map(model_groups).fillna("Other")

plot_metadata_df = metadata_df.copy()
plot_metadata_df["display_name"] = plot_metadata_df["run_name"].map(display_names).fillna(plot_metadata_df["run_name"].str.replace("_", " "))
plot_metadata_df["model_group"] = plot_metadata_df["run_name"].map(model_groups).fillna("Other")

plot_per_class_df = selected_per_class_df.copy()
plot_per_class_df["display_name"] = plot_per_class_df["run_name"].map(display_names).fillna(plot_per_class_df["run_name"].str.replace("_", " "))
plot_per_class_df["model_group"] = plot_per_class_df["run_name"].map(model_groups).fillna("Other")


### Few-Shot Curves by Model


In [ ]:
metrics_to_plot = [
    ("mean_auc_mean", "mean_auc_std", "Mean AUROC"),
    ("f1_macro_mean", "f1_macro_std", "Macro F1"),
    ("f1_micro_mean", "f1_micro_std", "Micro F1"),
    ("mean_accuracy_mean", None, "Mean Accuracy"),
    ("exact_match_accuracy_mean", None, "Exact Match Accuracy"),
]

fewshot_plot_df = plot_summary_df[plot_summary_df["setting"] != "full"].copy()

fig, axes = plt.subplots(2, 3, figsize=(17, 8))
axes = axes.ravel()

for ax, (mean_col, std_col, title) in zip(axes, metrics_to_plot):
    for display_name, group_df in fewshot_plot_df.groupby("display_name", sort=False):
        group_df = group_df.sort_values("n_train")
        x = group_df["n_train"].astype(float).to_numpy()
        y = group_df[mean_col].astype(float).to_numpy()

        ax.plot(x, y, marker="o", linewidth=2, label=display_name)

        if std_col is not None and group_df[std_col].notna().any():
            std = group_df[std_col].astype(float).fillna(0).to_numpy()
            runs = group_df["runs"].astype(float).to_numpy()
            ci95 = 1.96 * std / np.sqrt(runs)
            ax.fill_between(x, y - ci95, y + ci95, alpha=0.12)

    ax.set_xscale("log")
    ax.set_xlabel("Number of training/reference images")
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

axes[-1].axis("off")
handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="lower center", ncol=2)
fig.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()


### Full-Reference Metrics


In [ ]:
full_plot_df = plot_summary_df[plot_summary_df["setting"] == "full"].copy()
full_plot_df = full_plot_df.sort_values("mean_auc_mean", ascending=False)
model_order = full_plot_df["display_name"].tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

box_specs = [
    ("auc", "Per-Class AUROC Distribution"),
    ("f1", "Per-Class F1 Distribution (Macro-F1 Components)"),
]

for ax, (value_col, title) in zip(axes[:2], box_specs):
    box_values = [
        plot_per_class_df.loc[plot_per_class_df["display_name"] == name, value_col].dropna().to_numpy()
        for name in model_order
    ]
    ax.boxplot(box_values, labels=model_order, showmeans=True)
    ax.set_title(title)
    ax.set_ylabel(value_col.upper() if value_col == "auc" else "Per-class F1")
    ax.tick_params(axis="x", rotation=35)
    ax.grid(True, axis="y", alpha=0.3)

aggregate_metrics = [
    ("mean_auc_mean", "Mean AUROC"),
    ("f1_macro_mean", "Macro F1"),
    ("f1_micro_mean", "Micro F1"),
]
x = np.arange(len(model_order))
for metric_col, label in aggregate_metrics:
    y = full_plot_df.set_index("display_name").loc[model_order, metric_col].to_numpy()
    axes[2].plot(x, y, marker="o", linewidth=2, label=label)

axes[2].set_xticks(x)
axes[2].set_xticklabels(model_order, rotation=35, ha="right")
axes[2].set_title("Aggregate Full-Reference Metrics")
axes[2].set_ylabel("Metric value")
axes[2].grid(True, axis="y", alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

full_plot_df[
    [
        "display_name",
        "model_name",
        "image_size",
        "n_train",
        "k",
        "threshold",
        "mean_auc_mean",
        "f1_macro_mean",
        "f1_micro_mean",
        "mean_accuracy_mean",
        "exact_match_accuracy_mean",
    ]
]


### k/Threshold Search Diagnostics


In [ ]:
grid_plot_df = combined_summary_df.copy()
grid_plot_df["display_name"] = grid_plot_df["run_name"].map(display_names).fillna(grid_plot_df["run_name"].str.replace("_", " "))
grid_plot_df["model_group"] = grid_plot_df["run_name"].map(model_groups).fillna("Other")

# For each model/k in the full-reference setting, keep the best threshold.
full_best_threshold_per_k_df = (
    grid_plot_df[grid_plot_df["setting"] == "full"]
    .sort_values(["run_name", "k", selection_metric, "mean_auc_mean"], ascending=[True, True, False, False])
    .groupby(["run_name", "k"], as_index=False)
    .head(1)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for display_name, group_df in full_best_threshold_per_k_df.groupby("display_name", sort=False):
    group_df = group_df.sort_values("k")
    axes[0].plot(group_df["k"], group_df["f1_macro_mean"], marker="o", label=display_name)
    axes[1].plot(group_df["k"], group_df["mean_auc_mean"], marker="o", label=display_name)

axes[0].set_title("Full-Reference Macro F1 vs k")
axes[1].set_title("Full-Reference AUROC vs k")
for axis in axes:
    axis.set_xscale("log")
    axis.set_xlabel("k")
    axis.grid(True, alpha=0.3)
axes[0].set_ylabel("Macro F1")
axes[1].set_ylabel("Mean AUROC")
axes[1].legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

full_best_threshold_per_k_df[
    ["display_name", "k", "threshold", "mean_auc_mean", "f1_macro_mean", "f1_micro_mean", "mean_accuracy_mean"]
].sort_values(["display_name", selection_metric], ascending=[True, False])


### Efficiency vs Performance


In [ ]:
efficiency_df = full_plot_df.merge(
    plot_metadata_df,
    on=["dataset_name", "run_name", "model_name", "image_size", "display_name", "model_group"],
    how="left",
    suffixes=("", "_metadata"),
)

fig, axes = plt.subplots(2, 3, figsize=(17, 9), sharex="col")

scatter_specs = [
    ("total_params", "Total Parameters", True),
    ("efficiency_counted_seconds", "Counted Method Seconds", False),
    ("peak_gpu_memory_mb", "Peak GPU Memory MB", False),
]
metric_specs = [
    ("mean_auc_mean", "Full-Reference Mean AUROC", "AUROC"),
    ("f1_macro_mean", "Full-Reference Macro F1", "Macro F1"),
]

for row_idx, (metric_col, y_label, metric_title) in enumerate(metric_specs):
    for ax, (x_col, x_label, log_x) in zip(axes[row_idx], scatter_specs):
        plot_df = efficiency_df.dropna(subset=[x_col, metric_col])
        for model_group, group_df in plot_df.groupby("model_group", sort=False):
            ax.scatter(
                group_df[x_col],
                group_df[metric_col],
                s=90,
                color=group_colors.get(model_group, "tab:gray"),
                label=model_group,
                alpha=0.9,
                edgecolor="black",
                linewidth=0.5,
            )

        for idx, (_, row) in enumerate(plot_df.iterrows()):
            offset_y = 8 if idx % 2 == 0 else -12
            ax.annotate(
                row["display_name"],
                (row[x_col], row[metric_col]),
                textcoords="offset points",
                xytext=(6, offset_y),
                fontsize=8,
                alpha=0.9,
            )

        if log_x:
            ax.set_xscale("log")
        ax.set_xlabel(x_label)
        ax.set_ylabel(y_label)
        ax.set_title(f"{metric_title} vs {x_label}")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best", fontsize=8)

plt.tight_layout()
plt.show()

efficiency_df[
    [
        "run_name",
        "model_group",
        "mean_auc_mean",
        "f1_macro_mean",
        "f1_micro_mean",
        "k",
        "threshold",
        "total_params",
        "trainable_params",
        "trainable_param_fraction",
        "feature_dim",
        "feature_storage_mb",
        "feature_extraction_seconds",
        "adaptation_seconds",
        "inference_or_eval_seconds",
        "threshold_search_seconds",
        "efficiency_counted_seconds",
        "actual_wall_seconds",
        "peak_gpu_memory_mb",
        "shortcut_used",
    ]
]


### Per-Class Full-Reference Diagnostics


In [ ]:
full_per_class_plot_df = plot_per_class_df.copy()

per_class_table_df = full_per_class_plot_df[
    [
        "display_name",
        "k",
        "threshold",
        "class_name",
        "true_positive_rate",
        "predicted_positive_rate",
        "accuracy",
        "f1",
        "auc",
    ]
].sort_values(["display_name", "class_name"])

per_class_table_df


In [ ]:
if not full_per_class_plot_df.empty:
    class_order = (
        full_per_class_plot_df.groupby("class_name")["auc"]
        .mean()
        .sort_values(ascending=True)
        .index
        .tolist()
    )
    model_order = [name for name in model_order if name in full_per_class_plot_df["display_name"].unique()]

    fig, axes = plt.subplots(
        len(model_order),
        2,
        figsize=(15, 3.2 * len(model_order)),
        sharex="col",
        sharey=True,
    )
    axes = np.atleast_2d(axes)

    for row_idx, display_name in enumerate(model_order):
        model_df = full_per_class_plot_df[full_per_class_plot_df["display_name"] == display_name]
        model_df = model_df.set_index("class_name").loc[class_order].reset_index()

        color = group_colors.get(model_df["model_group"].iloc[0], "tab:gray")
        axes[row_idx, 0].barh(model_df["class_name"], model_df["auc"], color=color, alpha=0.85)
        axes[row_idx, 1].barh(model_df["class_name"], model_df["f1"], color=color, alpha=0.85)

        axes[row_idx, 0].set_ylabel(display_name)
        axes[row_idx, 0].grid(True, axis="x", alpha=0.3)
        axes[row_idx, 1].grid(True, axis="x", alpha=0.3)

    axes[0, 0].set_title("Per-Class AUROC")
    axes[0, 1].set_title("Per-Class F1 (Macro-F1 Components)")
    axes[-1, 0].set_xlabel("AUROC")
    axes[-1, 1].set_xlabel("Per-class F1")
    fig.suptitle("Full-Reference Per-Class Diagnostics Across Models", y=1.0)
    plt.tight_layout()
    plt.show()
